# 11 - SQLite Measurement Readers

IviumSoft's DataServer writes each measurement to a SQLite file
(`DataServer_*.idf.sqlite`) and keeps a catalog (`index.sqlite`) of them, by default
under `C:\\IviumStat\\DataServer\\measurements`. The `Tools` layer reads these
**read-only** and **WAL-safe**, so you can even tail a measurement while IviumSoft is
still writing it - and reading needs no driver or hardware.

### What you can do

- Browse the catalog and filter by serial number, technique or date
- Open a measurement and read its metadata, method, parts and points
- Decode the per-point status byte (overloads, current range)
- Read impedance (EIS) points
- Tail a running measurement incrementally (`after_point_id` / `latest_point_id`)
- Export to CSV (and to a pandas DataFrame if pandas is installed)

### Point it at your data

This notebook reads the **real measurement files on your machine**. Set `DATA_SERVER_DIR`
below to your DataServer measurements folder. Every data cell guards itself if the folder
is not present, so you can still read through the notebook without IviumSoft installed.

In [ ]:
from pathlib import Path

from pyvium.tools import MeasurementReader, MeasurementIndex
print("pyvium.tools SQLite readers imported")

## 1. Point at your DataServer

`index.sqlite` lives in the measurements folder; each measurement's
`DataServer_*.idf.sqlite` file is located by its catalog row (`path` + `file`), resolved
under this same folder. Change `DATA_SERVER_DIR` if your install differs.

In [ ]:
# Default IviumSoft location; edit for your machine.
DATA_SERVER_DIR = Path(r"C:\IviumStat\DataServer\measurements")
INDEX_PATH = DATA_SERVER_DIR / "index.sqlite"

HAVE_INDEX = INDEX_PATH.exists()
if HAVE_INDEX:
    print("using catalog:", INDEX_PATH)
else:
    print(f"No index.sqlite at {INDEX_PATH}")
    print("Edit DATA_SERVER_DIR above to point at your DataServer measurements folder.")

## 2. Browse the catalog

`MeasurementIndex` reads `index.sqlite` and returns entries newest-first. It filters by
serial (case-insensitive), technique, title substring, project/operator and date range.

In [ ]:
entry = None
if HAVE_INDEX:
    with MeasurementIndex(str(INDEX_PATH)) as index:
        recent = index.entries(limit=10)
        for e in recent:
            print(f"  {e.start_time}  {str(e.technique):<20} serial={e.serialnumber}  {e.file}")

        # Example filters (uncomment and adapt):
        # index.entries(serialnumber="P33162")
        # index.entries(technique="CyclicVoltammetry")
        # index.entries(start_after="2024-01-01", start_before="2024-12-31")

    entry = recent[0] if recent else None
    print("\nselected:", entry.file if entry else "catalog is empty")
else:
    print("skipped - no catalog (set DATA_SERVER_DIR in cell 1)")

## 3. Open the selected measurement

`resolve_path(entry, base_dir)` builds the file path from the catalog row;
`open_measurement(entry, base_dir)` hands back a (still unopened) `MeasurementReader`.
`base_dir` is the measurements folder.

In [ ]:
MEASUREMENT_PATH = None
if entry is not None:
    resolved = MeasurementIndex.resolve_path(entry, str(DATA_SERVER_DIR))
    print("measurement file:", resolved)
    if Path(resolved).exists():
        MEASUREMENT_PATH = resolved
        with MeasurementReader(MEASUREMENT_PATH) as reader:
            print("database version :", reader.database_version)
            print("metadata         :", reader.metadata())
            print("measurements     :", reader.measurements())
            print("method params    :", reader.method_parameters())
            for part in reader.measurement_parts()[:5]:
                print("  part:", part)
    else:
        print("File not found - adjust base_dir (resolve_path) to match your layout.")
else:
    print("no measurement selected")

## 4. Read points (with decoded status)

`read_points()` joins each point to its part context (cycle / level / channel) and
exposes a decoded `status` (overload flags and current-range index) from the `statusbyte`.

In [ ]:
if MEASUREMENT_PATH:
    with MeasurementReader(MEASUREMENT_PATH) as reader:
        points = reader.read_points()
    print(f"{len(points)} points; first 5:")
    for p in points[:5]:
        print(f"  id={p.point_id} t={p.t} x={p.x} y={p.y} z={p.z} "
              f"cycle={p.cycle} level={p.level} status={p.status}")
else:
    print("no measurement selected")

## 5. Tail a running measurement

While IviumSoft is still writing, poll `latest_point_id()` and pass the last id you
consumed to `read_points(after_point_id=...)` to fetch only the new points. The reader
opens read-only and WAL-safe, so it never blocks the writer.

In [ ]:
if MEASUREMENT_PATH:
    with MeasurementReader(MEASUREMENT_PATH) as reader:
        latest = reader.latest_point_id()
        print("latest point id:", latest)
        if latest and latest > 5:
            newer = reader.read_points(after_point_id=latest - 5)
            print(f"points after id {latest - 5}:", [p.point_id for p in newer])
else:
    print("no measurement selected")

## 6. Impedance (EIS) points

For EIS techniques the `pointfra` table holds the frequency response; `read_impedance()`
returns `ImpedancePoint`s. For non-EIS measurements (no `pointfra`) it returns an empty
list, so you need not know the technique in advance.

In [ ]:
if MEASUREMENT_PATH:
    with MeasurementReader(MEASUREMENT_PATH) as reader:
        eis = reader.read_impedance()
    if eis:
        for z in eis[:5]:
            print(f"  id={z.point_id}  f={z.frequency} Hz  Z'={z.z_re}  Z''={z.z_im}  "
                  f"quality={z.quality}")
    else:
        print("no impedance points (not an EIS measurement)")
else:
    print("no measurement selected")

## 7. Export

`to_csv()` writes all points; `to_dataframe()` returns a pandas DataFrame (pandas is
imported lazily, so it is only needed if you call it).

In [ ]:
if MEASUREMENT_PATH:
    import tempfile

    out_csv = Path(tempfile.gettempdir()) / "measurement_points.csv"
    with MeasurementReader(MEASUREMENT_PATH) as reader:
        reader.to_csv(str(out_csv))
    print("wrote", out_csv)
    for line in out_csv.read_text(encoding="utf-8").splitlines()[:3]:
        print("  ", line)

    try:
        with MeasurementReader(MEASUREMENT_PATH) as reader:
            dataframe = reader.to_dataframe()
        print(dataframe.head())
    except ImportError:
        print("pandas not installed - skipping to_dataframe()")
else:
    print("no measurement selected")

## 8. Shortcut: the last measurement, straight from IviumSoft

Instead of browsing the catalog, `Pyvium.get_db_file_name()` returns the full path of the
**most recently created** measurement DB. This one needs the driver open and IviumSoft
running (it is the only cell here that talks to the driver); the readers themselves do not.

In [ ]:
from pyvium import Pyvium

try:
    Pyvium.open_driver()
    db_path = Pyvium.get_db_file_name()
    print("last created DB:", db_path)
    if db_path and Path(db_path).exists():
        with MeasurementReader(db_path) as reader:
            print("technique:", reader.method_parameters().get("Technique"))
            print("points:", len(reader.read_points()))
except Exception as error:
    print(f"skipped ({type(error).__name__}: {error}) - needs IviumSoft running")
finally:
    try:
        Pyvium.close_driver()
    except Exception:
        pass

---

## Summary

| Task | API |
|------|-----|
| Open the catalog | `MeasurementIndex(index_path)` (default folder `C:\\IviumStat\\DataServer\\measurements`) |
| Browse / filter | `.entries(serialnumber=..., technique=..., start_after=..., limit=...)` |
| Entry -> file path | `MeasurementIndex.resolve_path(entry, base_dir)` |
| Entry -> reader | `.open_measurement(entry, base_dir)` |
| Open a measurement file | `MeasurementReader(path)` (context manager) |
| Metadata / method / parts | `.metadata()`, `.method_parameters()`, `.measurement_parts()` |
| Points (decoded status) | `.read_points()`, `point.status` |
| Tail a live run | `.latest_point_id()`, `.read_points(after_point_id=...)` |
| Impedance | `.read_impedance()` |
| Export | `.to_csv(path)`, `.to_dataframe()` |
| Last created DB (needs IviumSoft) | `Pyvium.get_db_file_name()` |

## Next

- **`10_instance_lifecycle_management.ipynb`** - launch, adopt and close IviumSoft instances
- **`07_data_processing.ipynb`** - parse IDF files and export to CSV